In [1]:
import os, sys
sys.path.insert(0, "/home/kmercad/mamba_har_2/project-basilisk")
from MambaSSL_JEPA_Model import MambaJEPA, HARMambaConfig
from data.OPPORTUNITY_data import load_OPP_loco_data, data_split_OPP, make_loaders_OPP
from utils import set_seed
from datetime import datetime
import torch
import torch.nn as nn
from tqdm import tqdm
import argparse
import time
sys.path.insert(0, "/home/kmercad/mamba_har_2/project-basilisk")


In [2]:
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True NVIDIA A100-SXM4-80GB


In [17]:
# OPPORTUNITY input:     [B, 90, 45]
# Conv1d + BN + GELU:    [B, 90, 384] -> obtaining 90 latent tokens (tokenizer)
# Mamba backbone:        [B, 90, 384]


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:

config = HARMambaConfig()
model = MambaJEPA(config, mask_ratio = 0.33).to(device)

B, L, Features = 2, 90, config.num_sensor_features
test_input = torch.rand((B, L, Features)).to(device)
target_embeddings, targets, context_embeddings, mask_targets, target_blocks, predictions = model(test_input)

print("target_embeddings: ", target_embeddings.shape)   # [2, 18, 384]
print("targets: ", targets.shape)             # [4, 3, 384]  (2B, t_l, d_model)
print("context_embeddings: ", context_embeddings.shape) # [2, 18, 384]
print("mask_targets: ", mask_targets.shape, "-", mask_targets[0].sum().item(), "masked of", mask_targets.shape[1])
print("target_blocks: ", len(target_blocks))
print("predictions: ", predictions.shape)         # [4, 3, 384]  must match targets

model.update_target_encoder(momentum = 0.996)
print("EMA OK")



target_embeddings:  torch.Size([2, 18, 384])
targets:  torch.Size([4, 3, 384])
context_embeddings:  torch.Size([2, 18, 384])
mask_targets:  torch.Size([2, 18]) - 6 masked of 18
target_blocks:  2
predictions:  torch.Size([4, 3, 384])
EMA OK


In [5]:
training_files, validation_files, test_files = data_split_OPP(1)
X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows = load_OPP_loco_data(training_files, validation_files, test_files, verbose = True)

------------------------------------------------------------------------------------------
Raw training set shape: (376092, 251)
Validation training set shape: (88848, 251)
Raw test set shape: (179695, 251)
------------------------------------------------------------------------------------------
Sensor subset training shape: (295149, 47)
Sensor Validation training shape: (67434, 47)
Sensor Test training shape: (134613, 47)
----------------------------------------------------------------------
Rows containing NaNs - training: 4953
Rows containing NaNs - validation: 0
Rows containing NaNs - test: 0
------------------------------------------------------------------------------------------
Training shape after NaN removal: (290196, 47)
Validation shape after NaN removal: (67434, 47)
Test shape after NaN removal: (134613, 47)
------------------------------------------------------------------------------------------
Reindexed training shape: (290196, 47)
Reindexed validation shape: (67434, 

In [6]:
#  ----------------------------------------------------- VALIDATION -----------------------------------------------------
@torch.no_grad()
def validate_model_PRETRAIN_JEPA(model, val_loader, device, criterion):
    '''
    Validation: avg latent prediction loss, with fixed masks each epoch.
    Also returns target-embedding std as a collapse monitor.
    '''    
    model.eval()
    total_loss = 0.0
    total_samples = 0
    emb_stds = []
    mean_coss = []
    
    with torch.random.fork_rng():
        torch.manual_seed(42)
        for x_batch, _ in val_loader:
            x_batch = x_batch.to(device, non_blocking = True)
            target_embeddings, targets, _, _, _, predictions = model(x_batch)
            loss = criterion (predictions, targets)

            bsize = x_batch.size(0)
            total_loss += loss.item() * bsize
            total_samples += bsize

            flat_emb = target_embeddings.reshape(-1, target_embeddings.size(-1)) # [B*18, 384]
            #Monitor 1: per ft std across all tkns (collapse -> 0)
            emb_stds.append(flat_emb.std(dim = 0).mean().item())
            #Monitor 2: avg cosine similarity bt tokens (collapse -> 1)
            normed = torch.nn.functional.normalize(flat_emb, dim = -1)
            mean_coss.append(normed.mean(dim = 0).norm().pow(2).item())
 
    return total_loss / total_samples, sum(emb_stds) / len(emb_stds), sum(mean_coss) / len(mean_coss)

In [7]:
#  ---------------------------------------------------- TRAINING ---------------------------------------------------
for seed in [42, 58, 7, 128, 92]:
    g = set_seed(seed)
    train_loader, val_loader, test_loader, label_encoder = make_loaders_OPP(X_windows, y_windows, X_validation_windows, y_validation_windows, X_test_windows, y_test_windows, generator = g, verbose = True)

    #  ----------- TRAINING SETUP -----------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = HARMambaConfig()
    model = MambaJEPA(config, mask_ratio = 1/3, t_l = 3)
    model.to(device, non_blocking = True)
    # ----------------------
    num_epochs = 50
    lr = 0.0006
    patience = 8
    criterion = nn.SmoothL1Loss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]   # excludes frozen target encoder
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: #skip frozen targer encoder
            continue
        if p.ndim <= 1 or name.endswith("pe") or "predictor_pe" in name or "mask_token" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    optimizer = torch.optim.AdamW([{"params": decay,    "weight_decay": 1e-4}, {"params": no_decay, "weight_decay": 0.0}], lr = lr)
    # EMA momentum schedule: 0.996 -> 1.0 linearly over all steps (I-JEPA style)
    total_steps = num_epochs * len(train_loader)
    m_start, m_end = 0.996, 1.0
    global_step = 0

    #  ----------- TRAINING -----------
    model_name = f"JEPA_models_pt/JEPA_model_OPP_fold{1}_seed{seed}.pt"
    epoch_history = []
    best_val_loss = float("inf")
    best_epoch = None
    best_state = None
    bad_epochs = 0
    with open(f"logs/JEPA_training_OPP_fold{1}.txt", "a") as log_file:
        log_file.write(f"\nTRAINING STARTING AT: {datetime.now()}\n")
        log_file.write(f"Model: {model_name} | SEED: {seed}\n")
        log_file.flush()
        for epoch in range(num_epochs):
            epoch_start = time.time()
            model.train()
            total_loss = 0.0
            total_samples = 0

            loop = tqdm(train_loader, desc = f"Epoch {epoch+1}/{num_epochs}")
            for _, (x_batch, _) in enumerate(loop):
                x_batch = x_batch.to(device, non_blocking = True)

                optimizer.zero_grad()
                _, targets, _, _, _, predictions = model(x_batch)
                loss = criterion(predictions, targets)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm = 1.0)
                optimizer.step()

                momentum = m_start + (m_end - m_start) * (global_step / total_steps)
                model.update_target_encoder(momentum = momentum)
                global_step += 1

                bsize = x_batch.size(0)
                total_loss += loss.item() * bsize
                total_samples += bsize
                loop.set_postfix(loss = f"{total_loss/total_samples:.4f}")

            train_loss = total_loss / total_samples
            val_loss, emb_std, mean_cos = validate_model_PRETRAIN_JEPA(model, val_loader, device, criterion)

            epoch_time = time.time() - epoch_start
            epoch_history.append({
                "epoch": epoch + 1,
                "tr_loss": float(train_loss),
                "val_loss": float(val_loss),
                "emb_std": float(emb_std),
                "mean_cos": float(mean_cos)
            })
            print(f"\nEpoch: {epoch+1}/{num_epochs} | tr_Loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | emb_std: {emb_std:.4f} | mean_cos: {mean_cos:.4f} | epoch_time: {epoch_time:.2f}s")
            log_file.write(f"Epoch: {epoch+1}/{num_epochs} | tr_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | emb_std: {emb_std:.4f} | mean_cos: {mean_cos:.4f} | epoch_time: {epoch_time:.2f}s\n")
            log_file.flush()

            if val_loss < best_val_loss - 1e-12:
                best_val_loss = float(val_loss)
                best_epoch = epoch + 1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"\nEarly stopping at epoch {epoch + 1} | Best Validation Loss: {best_val_loss:.4f}")
                    break

        if best_state is not None:
            torch.save(best_state, model_name)
        log_file.write(f"TRAINING ENDING AT: {datetime.now()}\n")

------------------------------------------------------------------------------------------
Training set class distribution:
{'STAND': 4195, 'WALK': 2640, 'SIT': 2295, 'LIE': 514}
------------------------------------------------------------------------------------------


Epoch 1/50: 100%|███████████████████████████████████████████████████████████████████████████████████████| 76/76 [00:25<00:00,  3.01it/s, loss=0.2284]



Epoch: 1/50 | tr_Loss: 0.2284 | val_loss: 0.1203 | emb_std: 0.6335 | mean_cos: 0.5636 | epoch_time: 27.03s


Epoch 2/50: 100%|███████████████████████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.05it/s, loss=0.0908]



Epoch: 2/50 | tr_Loss: 0.0908 | val_loss: 0.0816 | emb_std: 0.6702 | mean_cos: 0.5059 | epoch_time: 14.21s


...


Epoch: 7/50 | tr_Loss: 0.0347 | val_loss: 0.0365 | emb_std: 0.7612 | mean_cos: 0.3721 | epoch_time: 14.21s


Epoch 8/50: 100%|██████████████████████████████████████████████████████████████████████| 76/76 [00:12<00:00,  6.08it/s, loss=0.0314]



Epoch: 8/50 | tr_Loss: 0.0314 | val_loss: 0.0339 | emb_std: 0.7736 | mean_cos: 0.3510 | epoch_time: 14.15s


Epoch 9/50:   5%|███▋                                                                   | 4/76 [00:00<00:15,  4.74it/s, loss=0.0298]


KeyboardInterrupt: 